In [1]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import LoraConfig
from trl import SFTTrainer

In [2]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

CUDA Available: True
GPU: NVIDIA GeForce RTX 5060 Ti
VRAM GB: 15.92828369140625


In [3]:
train_dataset = load_dataset(
    "json",
    data_files="../data/qwen7b_scanner_train.jsonl",
    split="train"
)

val_dataset = load_dataset(
    "json",
    data_files="../data/qwen7b_scanner_val.jsonl",
    split="train"
)

print(train_dataset)
print(val_dataset)
print(train_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 960
})
Dataset({
    features: ['messages'],
    num_rows: 240
})
{'messages': [{'role': 'user', 'content': 'You are an OSS compliance risk analysis assistant.\n\nClassify this scanner output.\n\nPackage: node-package-low-225\nVersion: 4.19.19\nEcosystem: node\nPackage Manager: npm\nLicense: MIT\nLicense Family: Permissive\nDependency Scope: test\nDirect or Transitive: direct\nDevelopment Dependency: no\nProject Type: AI inference platform\nDistribution Model: internal-only\nUsage: data processing module\nLinking Type: static\nNetwork Exposed: yes\nCommercial Use: yes\nAttribution Notice: preserved\nLicense Text: included\nSource Modified: no\nRedistribution: no\nLicense Confidence: high\n\nReturn exactly in this format:\nRisk: <Low|Medium|High>\nReason: <one short sentence>'}, {'role': 'assistant', 'content': 'Risk: Low\nReason: MIT is low risk because obligations are satisfied and required notices or license text are preserved.'}]}


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [5]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Qwen2.5-7B loaded in 4-bit mode.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B loaded in 4-bit mode.


In [6]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(peft_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'o_proj', 'gate_proj', 'v_proj', 'k_proj', 'up_proj', 'q_proj', 'down_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [10]:
training_args = TrainingArguments(
    output_dir="../models/qwen7b_oss_compliance_lora",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=5e-5,
    num_train_epochs=3,

    logging_steps=10,

    save_strategy="epoch",
    eval_strategy="epoch",

    fp16=False,
    bf16=False,

    max_grad_norm=0.0,

    report_to="none",

    optim="paged_adamw_8bit"
)

In [11]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    args=training_args
)

print("Trainer created.")

/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Trainer created.


In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.170724,0.171339
2,0.160615,0.161668
3,0.158786,0.159868


TrainOutput(global_step=360, training_loss=0.30783918797969817, metrics={'train_runtime': 1604.0517, 'train_samples_per_second': 1.795, 'train_steps_per_second': 0.224, 'total_flos': 2.6644791620192256e+16, 'train_loss': 0.30783918797969817})

In [13]:
trainer.save_model("../models/qwen7b_oss_compliance_lora")
print("Qwen2.5-7B OSS compliance LoRA saved.")

Qwen2.5-7B OSS compliance LoRA saved.
